In [ ]:
import json
import re
import random
from pathlib import Path

DATA_PATH = Path("intents.json")
MEMORY_PATH = Path("memory.json")


In [ ]:
with open(DATA_PATH, "r") as f:
    intents = json.load(f)["intents"]

if MEMORY_PATH.exists():
    memory = json.loads(MEMORY_PATH.read_text())
else:
    memory = {}

compiled_intents = []
for intent in intents:
    pattern_list = [re.compile(p, re.I) for p in intent["patterns"]]
    compiled_intents.append({
        "tag": intent["tag"],
        "patterns": pattern_list,
        "responses": intent["responses"]
    })

In [ ]:
anime_recommendations = {
    "action": ["Jujutsu Kaisen", "Attack on Titan", "Demon Slayer", "Vinland Saga"],
    "romance": ["Your Name", "Horimiya", "Toradora", "Kimi ni Todoke"],
    "comedy": ["Gintama", "KonoSuba", "Saiki K", "Nichijou"],
    "fantasy": ["Re:Zero", "Made in Abyss", "Magi", "Mushoku Tensei"],
    "isekai": ["Re:Zero", "Konosuba", "Tensei Slime", "Mushoku Tensei"],
    "slice of life": ["Clannad", "Barakamon", "Anohana", "March Comes in Like a Lion"],
    "horror": ["Tokyo Ghoul", "Parasyte", "Another", "Elfen Lied"],
    "psychological": ["Death Note", "Steins;Gate", "Monster", "Erased"],
    "sports": ["Haikyuu!!", "Kuroko no Basket", "Blue Lock", "Yuri on Ice"],
    "mecha": ["Code Geass", "Neon Genesis Evangelion", "Gurren Lagann", "86"]
}


In [ ]:
def save_memory():
    with open(MEMORY_PATH, "w") as f:
        json.dump(memory, f)

In [ ]:
def match_intent(user_input):
    # REGEX CHECK
    for intent in compiled_intents:
        for pattern in intent["patterns"]:
            match = pattern.search(user_input)

            if match:
                response = random.choice(intent["responses"])

                # Recommendations
                if intent["tag"] == "ask_genre":
                    genre = match.group(1).lower()
                    recs = anime_recommendations.get(
                        genre, ["No anime available for that genre."]
                    )
                    recommendation = ", ".join(recs)
                    response = response.replace("{recommendation}", recommendation)
                    response = response.format(genre)

                # Genre
                if intent["tag"] == "genres":
                    genre = match.group(0).lower()
                    recs = anime_recommendations.get(
                        genre, ["No suggestions yet!"]
                    )
                    recommendation = ", ".join(recs)
                    response = response.replace("{0}", genre)

                # Named groups
                if match.groupdict():
                    response = response.format(**match.groupdict())
                    if "name" in match.groupdict():
                        memory["name"] = match.group("name")
                        save_memory()

                # Numbered groups
                elif match.groups():
                    response = response.format(*match.groups())

                return intent["tag"], [response]

    return None, ["Sorry, I didn’t understand that. Try again?"]

In [ ]:
print("AnimeBot is ready! Type 'exit' to quit.")

while True:
    user = input("You: ").strip()

    if user.lower() in ["exit", "quit"]:
        print("AnimeBot: Bye Bye!")
        break

    if user.lower() == "show memory":
        print("AnimeBot:", memory)
        continue

    tag, responses = match_intent(user)
    print("AnimeBot:", random.choice(responses))

testing outputs


In [ ]:
# Step 1: Greetings
print(match_intent("hi"))
print(match_intent("hello"))

# Step 2: Genre recommendations
print(match_intent("recommend action anime"))
print(match_intent("recommend romance anime"))

# Step 3: Genre keywords
print(match_intent("comedy"))
print(match_intent("fantasy"))

# Step 4: Memory storage and recall
print(match_intent("my favourite anime is One Piece"))
print(match_intent("what anime do i like"))

# Step 5: Top anime and goodbye
print(match_intent("top anime"))
print(match_intent("goodbye"))